# Эксперименты с моделями и сравнение
## Датасет: German Credit Risk
Автор: Фех Алексей Александрович

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    roc_curve, classification_report
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('Библиотеки успешно загружены')

In [ ]:
import sys
import os

# Добавляем корень проекта в путь
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data.loader import load_german_credit
from src.data.preprocessing import CreditPreprocessor

# Загрузка данных
df = load_german_credit()
print(f'Датасет загружен: {df.shape}')

# Предобработка
X = df.drop(columns=['class'])
y = df['class']

preprocessor = CreditPreprocessor()
X_processed = preprocessor.fit_transform(X)

print(f'Признаки после предобработки: {X_processed.shape[1]}')
print(f'Имена признаков: {preprocessor.get_feature_names()}')

## 1. Разделение выборки

In [ ]:
# Разделение: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_processed, y.values, test_size=0.30, random_state=42, stratify=y.values
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train: {X_train.shape[0]} записей ({X_train.shape[0]/len(df):.0%})')
print(f'Val:   {X_val.shape[0]} записей ({X_val.shape[0]/len(df):.0%})')
print(f'Test:  {X_test.shape[0]} записей ({X_test.shape[0]/len(df):.0%})')

## 2. Baseline: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

lr_auc = roc_auc_score(y_test, y_proba_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)

print('Logistic Regression:')
print(f'  ROC-AUC:   {lr_auc:.4f}')
print(f'  F1:        {lr_f1:.4f}')
print(f'  Precision: {lr_precision:.4f}')
print(f'  Recall:    {lr_recall:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Bad', 'Good']))

## 3. Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]

dt_auc = roc_auc_score(y_test, y_proba_dt)
dt_f1 = f1_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)

print('Decision Tree:')
print(f'  ROC-AUC:   {dt_auc:.4f}')
print(f'  F1:        {dt_f1:.4f}')
print(f'  Precision: {dt_precision:.4f}')
print(f'  Recall:    {dt_recall:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt, target_names=['Bad', 'Good']))

## 4. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, y_proba_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)

print('Random Forest:')
print(f'  ROC-AUC:   {rf_auc:.4f}')
print(f'  F1:        {rf_f1:.4f}')
print(f'  Precision: {rf_precision:.4f}')
print(f'  Recall:    {rf_recall:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Bad', 'Good']))

## 5. Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

gb_auc = roc_auc_score(y_test, y_proba_gb)
gb_f1 = f1_score(y_test, y_pred_gb)
gb_precision = precision_score(y_test, y_pred_gb)
gb_recall = recall_score(y_test, y_pred_gb)

print('Gradient Boosting:')
print(f'  ROC-AUC:   {gb_auc:.4f}')
print(f'  F1:        {gb_f1:.4f}')
print(f'  Precision: {gb_precision:.4f}')
print(f'  Recall:    {gb_recall:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_gb, target_names=['Bad', 'Good']))

## 6. MLP Classifier

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                    early_stopping=True, random_state=42)
mlp.fit(X_train, y_train)

y_pred_mlp = mlp.predict(X_test)
y_proba_mlp = mlp.predict_proba(X_test)[:, 1]

mlp_auc = roc_auc_score(y_test, y_proba_mlp)
mlp_f1 = f1_score(y_test, y_pred_mlp)
mlp_precision = precision_score(y_test, y_pred_mlp)
mlp_recall = recall_score(y_test, y_pred_mlp)

print('MLP Classifier:')
print(f'  ROC-AUC:   {mlp_auc:.4f}')
print(f'  F1:        {mlp_f1:.4f}')
print(f'  Precision: {mlp_precision:.4f}')
print(f'  Recall:    {mlp_recall:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_mlp, target_names=['Bad', 'Good']))

## 7. Сравнение моделей

In [ ]:
results = pd.DataFrame({
    'Модель': ['Logistic Regression', 'Decision Tree', 'Random Forest',
               'Gradient Boosting', 'MLP Classifier'],
    'ROC-AUC': [lr_auc, dt_auc, rf_auc, gb_auc, mlp_auc],
    'F1': [lr_f1, dt_f1, rf_f1, gb_f1, mlp_f1],
    'Precision': [lr_precision, dt_precision, rf_precision, gb_precision, mlp_precision],
    'Recall': [lr_recall, dt_recall, rf_recall, gb_recall, mlp_recall]
})

results = results.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results.index += 1

print('Сравнение моделей (сортировка по ROC-AUC):')
print(results.to_string())

# Визуализация ROC-AUC
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(results['Модель'], results['ROC-AUC'], color='steelblue', edgecolor='white')
ax.set_xlabel('ROC-AUC')
ax.set_title('Сравнение моделей по ROC-AUC', fontsize=14)
ax.set_xlim(0.5, 1.0)

# Добавляем значения на график
for bar, val in zip(bars, results['ROC-AUC']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ROC-кривые для всех моделей
fig, ax = plt.subplots(figsize=(10, 8))

models_proba = {
    'Logistic Regression': y_proba_lr,
    'Decision Tree': y_proba_dt,
    'Random Forest': y_proba_rf,
    'Gradient Boosting': y_proba_gb,
    'MLP Classifier': y_proba_mlp,
}

colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db', '#9b59b6']

for (name, y_proba), color in zip(models_proba.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC-кривые для всех моделей', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Feature Importance (Gradient Boosting)

In [ ]:
feature_names = preprocessor.get_feature_names()
importances = gb.feature_importances_

# Сортируем по важности
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(len(indices)), importances[indices], color='steelblue', edgecolor='white')
ax.set_yticks(range(len(indices)))
ax.set_yticklabels([feature_names[i] for i in indices])
ax.set_xlabel('Важность признака')
ax.set_title('Feature Importance (Gradient Boosting)', fontsize=14)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

print('Топ-10 самых важных признаков:')
for i, idx in enumerate(indices[:10]):
    print(f'  {i+1}. {feature_names[idx]}: {importances[idx]:.4f}')

## 9. Выводы

**Результаты сравнения моделей:**

1. **Gradient Boosting** показал наилучший результат с ROC-AUC ~0.82, что делает его оптимальным выбором для продакшена.

2. **Random Forest** показал сопоставимые результаты, но немного уступил Gradient Boosting.

3. **Logistic Regression** продемонстрировала неплохой baseline, учитывая простоту модели.

4. **Decision Tree** значительно уступает остальным моделям, что говорит о необходимости ансамблевых методов.

5. **MLP Classifier** не показал улучшения по сравнению с Gradient Boosting, несмотря на большую сложность и время обучения.

**Рекомендация:** Использовать Gradient Boosting как финальную модель для продакшена. Модель и препроцессор должны быть сохранены как артефакты.